# Multi-Physics Climate Modeling

## Level 1 Simple Atmosphere Model

This challenge guides you to implement a 2D simple atmosphere model using PhysicsNeMo and PINNs. We consider a scalar temperature field T(x,y,t) transported by prescribed horizontal winds, diffused by subgrid processes, and relaxed to an equilibrium state.

### Mathematical Formulation
We model temperature evolution with a linear advection–diffusion–reaction equation:

$$ T_t + u T_x + v T_y = \kappa (T_{xx} + T_{yy}) + Q - \lambda (T - T_{eq}) $$

where:
- $u(x,y,t), v(x,y,t)$: horizontal wind components (constants in this challenge)
- $\kappa$: thermal diffusivity (constant)
- $Q(x,y,t)$: heat source (constant or zero)
- $\lambda$: Newtonian cooling rate to equilibrium temperature $T_{eq}$

### Problem Setup

For validation, we use a special case with:
- **No advection**: $u=v=0$
- **No source**: $Q=0$
- **No relaxation**: $\lambda=0$
- **Domain**: Square $[0,\pi]\times[0,\pi]$
- **Boundary conditions**: Dirichlet $T=0$ on all edges
- **Initial condition**: $T(x,y,0)=\sin(x)\sin(y)$
- **Parameters**: $\kappa=1.0$

The exact solution is:
$$T(x,y,t) = \sin(x)\sin(y)e^{-2\kappa t}$$

### Implementation Steps

#### 1. Import Required Libraries
```python
import os
import warnings

import numpy as np
from sympy import Symbol, Function, Number, sin

import physicsnemo.sym
from physicsnemo.sym.hydra import to_yaml, to_absolute_path, instantiate_arch
from physicsnemo.sym.hydra.config import PhysicsNeMoConfig

from physicsnemo.sym.solver import Solver
from physicsnemo.sym.domain import Domain
from physicsnemo.sym.geometry.primitives_2d import Rectangle
from physicsnemo.sym.domain.constraint import (
    PointwiseBoundaryConstraint,
    PointwiseInteriorConstraint,
)

from physicsnemo.sym.domain.validator import PointwiseValidator
from physicsnemo.sym.key import Key
from physicsnemo.sym.node import Node

from physicsnemo.sym.eq.pde import PDE
from physicsnemo.sym.utils.io import ValidatorPlotter

import matplotlib.pyplot as plt
plt.rcParams['image.cmap'] = 'jet'
```

#### 2. Define the Simple Atmosphere PDE
Create a class that inherits from `PDE` to define the 2D advection–diffusion–reaction model for temperature T(x,y,t).

```python
class SimpleAtmosphere2D(PDE):
    name = "SimpleAtmosphere2D"

    def __init__(self, u0=0.0, v0=0.0, kappa=1.0, lam=0.0, Q0=0.0, Teq=0.0):
        # Coordinates
        x = Symbol("x")
        y = Symbol("y")
        t = Symbol("t")

        # Input variables
        input_variables = {"x": x, "y": y, "t": t}

        # Temperature field
        T = Function("T")(*input_variables)

        # Helper to allow constants or functions
        def _as_const_or_func(val, name):
            if isinstance(val, str):
                return Function(name)(*input_variables)
            elif isinstance(val, (float, int)):
                return Number(val)
            else:
                return val

        # Parameters (constants for this challenge)
        u = _as_const_or_func(u0, "u")
        v = _as_const_or_func(v0, "v")
        k = _as_const_or_func(kappa, "kappa")
        lam = _as_const_or_func(lam, "lam")
        Q = _as_const_or_func(Q0, "Q")
        Teq = _as_const_or_func(Teq, "Teq")

        # PDE residual: T_t + u*T_x + v*T_y - k*(T_xx + T_yy) - Q + lam*(T - Teq) = 0
        self.equations = {}
        # Hint: Define the ADR residual using T.diff(...)
        self.equations["adr"] = FIXME # Fill in residual expression
```

#### 3. Set Up the Solver and Constraints

```python
@physicsnemo.sym.main(config_path="conf", config_name="config_atmos")
def run(cfg: PhysicsNeMoConfig) -> None:
    # Parameters for exact-solution case
    u0 = FIXME # Fill in
    v0 = FIXME # Fill in
    kappa = FIXME # Fill in
    lam = FIXME # Fill in
    Q0 = FIXME # Fill in
    Teq_val = FIXME # Fill in

    pde = SimpleAtmosphere2D(u0=u0, v0=v0, kappa=kappa, lam=lam, Q0=Q0, Teq=Teq_val)

    # Neural network
    atmos_net = instantiate_arch(
        input_keys=[Key("x"), Key("y"), Key("t")],
        output_keys=[Key("T")],
        cfg=cfg.arch.fully_connected,
    )

    # Nodes
    nodes = pde.make_nodes() + [atmos_net.make_node(name="atmos_network")]

    # Geometry and time
    x, y, t_symbol = Symbol("x"), Symbol("y"), Symbol("t")
    L = float(np.pi)
    geo = Rectangle((0.0, 0.0), (L, L))
    time_range = {t_symbol: (0.0, 2 * L)}

    # Domain
    domain = Domain()

    # Initial condition: T(x,y,0) = sin(x)*sin(y)
    IC = PointwiseInteriorConstraint(
        nodes=nodes,
        geometry=geo,
        outvar={
            FIXME # # Fill in: Add initial condition here for "T"
        },
        batch_size=cfg.batch_size.IC,
        lambda_weighting={"T": 1.0},
        parameterization={t_symbol: 0.0},
    )
    domain.add_constraint(IC, "IC")

    # Boundary condition: T=0 on all boundaries
    BC = PointwiseBoundaryConstraint(
        nodes=nodes,
        geometry=geo,
        outvar={
            FIXME # Fill in: Add boundary condition here for "T"
        },
        lambda_weighting={"T": 1.0},
        batch_size=cfg.batch_size.BC,
        parameterization=time_range,
    )
    domain.add_constraint(BC, "BC")

    # PDE interior constraint: enforce residual = 0
    interior = PointwiseInteriorConstraint(
        nodes=nodes,
        geometry=geo,
        outvar={
            FIXME # Fill in: Add PDE constraint here
        },
        batch_size=cfg.batch_size.interior,
        parameterization=time_range,
    )
    domain.add_constraint(interior, "interior")

    # Validation dataset with analytical solution
    deltaT = 0.01
    deltaX = 0.05
    deltaY = 0.05
    x_vals = np.arange(0.0, L, deltaX)
    y_vals = np.arange(0.0, L, deltaY)
    t_vals = np.arange(0.0, 2 * L, deltaT)
    X, Y, TT = np.meshgrid(x_vals, y_vals, t_vals)
    X = np.expand_dims(X.flatten(), axis=-1)
    Y = np.expand_dims(Y.flatten(), axis=-1)
    TT = np.expand_dims(TT.flatten(), axis=-1)

    # Exact: T(x,y,t) = sin(x) sin(y) exp(-2*kappa*t)
    T_true = FIXME # Fill in: Exact T

    invar_numpy = {"x": X, "y": Y, "t": TT}
    outvar_numpy = {"T": T_true}

    validator = PointwiseValidator(
        nodes=nodes,
        invar=invar_numpy,
        true_outvar=outvar_numpy,
        batch_size=128,
        plotter=ValidatorPlotter(),
    )
    domain.add_validator(validator)

    # Solver
    slv = Solver(cfg, domain)
    slv.solve()
```

1. Fill in all missing parts
2. Place the configuration file in the conf directory
3. Run the script:

In [ ]:
!python climate_l1.py

/usr/local/lib/python3.12/dist-packages/hydra/_internal/hydra.py:119: UserWarning: Future Hydra versions will no longer change working directory at job runtime by default.
See https://hydra.cc/docs/1.2/upgrades/1.1_to_1.2/changes_to_job_working_dir/ for more information.
  ret = run_job(
[W311 05:45:44.256283657 init.cpp:779] Warning: nvfuser is no longer supported in torch script, use _jit_set_nvfuser_single_node_mode is deprecated and a no-op (function operator())
[W311 05:45:44.256363308 init.cpp:767] Warning: nvfuser is no longer supported in torch script, use _jit_set_nvfuser_enabled is deprecated and a no-op (function operator())
[05:45:44] - JIT using the NVFuser TorchScript backend
[05:45:44] - JitManager: {'_enabled': True, '_arch_mode': <JitArchMode.ONLY_ACTIVATION: 1>, '_use_nvfuser': True, '_autograd_nodes': False}
[05:45:44] - GraphManager: {'_func_arch': False, '_debug': False, '_func_arch_allow_partial_hessian': True}
[05:45:44] - AmpManager: {'_enabled': False, '_mode':

## Level 2 Coupled Atmosphere-Ocean System

In this extension, we couple the atmospheric temperature field `Ta(x,y,t)` to an ocean mixed-layer temperature field `To(x,y,t)` through a linear heat exchange term with coupling strength `gamma`.

### Mathematical Formulation

We use an advection–diffusion–reaction model for the atmosphere and a diffusion model for the ocean mixed layer:

$$ T_{a,t} + u T_{a,x} + v T_{a,y} = \kappa_a (T_{a,xx} + T_{a,yy}) + Q_a - \lambda_a (T_a - T_{eq,a}) - \gamma (T_a - T_o) $$
$$T_{o,t} = \kappa_o (T_{o,xx} + T_{o,yy}) + Q_o + \gamma (T_a - T_o) \quad (\text{no advection in ocean}).$$

- `u, v`: prescribed winds (constants)
- `κ_a, κ_o`: diffusivities
- `Q_a, Q_o`: sources
- `λ_a`: Newtonian cooling to `T_eq,a`
- `γ`: atmosphere–ocean coupling coefficient

### Problem Setup

For validation, we use a decoupled case:
- **No advection**: $u=v=0$
- **No sources**: $Q_a=Q_o=0$
- **No relaxation**: $\lambda_a=0$
- **No coupling**: $\gamma=0$
- **Domain**: Square $[0,\pi]^2$
- **Boundary conditions**: $T_a=T_o=0$ on all edges
- **Initial conditions**: $T_a(x,y,0)=T_o(x,y,0)=\sin(x)\sin(y)$
- **Parameters**: $\kappa_a=1.0$, $\kappa_o=0.5$

The exact solutions are:
$$T_a(x,y,t) = \sin(x)\sin(y)e^{-2\kappa_a t}, \quad T_o(x,y,t) = \sin(x)\sin(y)e^{-2\kappa_o t}$$

### Implementation Steps

#### 1. Define the Coupled PDE
```python
class CoupledAtmosOcean2D(PDE):
    name = "CoupledAtmosOcean2D"

    def __init__(
        self,
        u0=0.0,
        v0=0.0,
        kappa_a=1.0,
        kappa_o=0.5,
        lam_a=0.0,
        Q_a0=0.0,
        Q_o0=0.0,
        Teq_a0=0.0,
        gamma0=0.0,
    ):
        # Coordinates
        x = Symbol("x")
        y = Symbol("y")
        t = Symbol("t")

        # Inputs
        input_variables = {"x": x, "y": y, "t": t}

        # Fields
        Ta = Function("Ta")(*input_variables)
        To = Function("To")(*input_variables)

        # Helper: allow constants or functions
        def _as_const_or_func(val, name):
            if isinstance(val, str):
                return Function(name)(*input_variables)
            elif isinstance(val, (float, int)):
                return Number(val)
            else:
                return val

        # Parameters
        u = _as_const_or_func(u0, "u")
        v = _as_const_or_func(v0, "v")
        ka = _as_const_or_func(kappa_a, "kappa_a")
        ko = _as_const_or_func(kappa_o, "kappa_o")
        lam = _as_const_or_func(lam_a, "lam_a")
        Qa = _as_const_or_func(Q_a0, "Q_a")
        Qo = _as_const_or_func(Q_o0, "Q_o")
        Teq_a = _as_const_or_func(Teq_a0, "Teq_a")
        gamma = _as_const_or_func(gamma0, "gamma")

        # Residuals (set to zero):
        # atm: Ta_t + u Ta_x + v Ta_y - ka*(Ta_xx+Ta_yy) - Qa + lam*(Ta-Teq_a) + gamma*(Ta-To) = 0
        # ocn: To_t - ko*(To_xx+To_yy) - Qo - gamma*(Ta-To) = 0
        self.equations = {}
        self.equations["atm"] = FIXME # Fill in atmospheric residue
        self.equations["ocn"] = FIXME # Fill in ocean residue
```

#### 2. Solver, Constraints, and Validator
```python
@physicsnemo.sym.main(config_path="conf", config_name="config_coupled")
def run_coupled(cfg: PhysicsNeMoConfig) -> None:
    # Parameters for validation case (decoupled)
    u0 = FIXME # Fill in
    v0 = FIXME # Fill in
    kappa_a = FIXME # Fill in
    kappa_o = FIXME # Fill in
    lam_a = FIXME # Fill in
    Q_a0 = FIXME # Fill in
    Q_o0 = FIXME # Fill in
    Teq_a0 = FIXME # Fill in
    gamma0 = FIXME # Fill in

    pde = CoupledAtmosOcean2D(
        u0=u0, v0=v0, kappa_a=kappa_a, kappa_o=kappa_o,
        lam_a=lam_a, Q_a0=Q_a0, Q_o0=Q_o0, Teq_a0=Teq_a0, gamma0=gamma0,
    )

    # Network with two outputs: Ta, To
    net = instantiate_arch(
        input_keys=[Key("x"), Key("y"), Key("t")],
        output_keys=[Key("Ta"), Key("To")],
        cfg=cfg.arch.fully_connected,
    )

    nodes = pde.make_nodes() + [net.make_node(name="coupled_network")]

    # Geometry and time
    x, y, t_symbol = Symbol("x"), Symbol("y"), Symbol("t")
    L = float(np.pi)
    geo = Rectangle((0.0, 0.0), (L, L))
    time_range = {t_symbol: (0.0, 2 * L)}

    domain = Domain()

    # Initial condition: Ta(x,y,0) = sin(x)*sin(y), To(x,y,0) = sin(x)*sin(y)
    IC = PointwiseInteriorConstraint(
        nodes=nodes,
        geometry=geo,
        outvar={
            FIXME # Fill in: Add initial condition here for "Ta", "To"
        },
        batch_size=cfg.batch_size.IC,
        lambda_weighting={"Ta": 1.0, "To": 1.0},
        parameterization={t_symbol: 0.0},
    )
    domain.add_constraint(IC, "IC")

    # Boundary condition: Ta=0, To=0 on all boundaries
    BC = PointwiseBoundaryConstraint(
        nodes=nodes,
        geometry=geo,
        outvar={
            FIXME # Fill in: Add boundary condition here for "Ta", "To"
        },
        lambda_weighting={"Ta": 1.0, "To": 1.0},
        batch_size=cfg.batch_size.BC,
        parameterization=time_range,
    )
    domain.add_constraint(BC, "BC")

    # PDE interior constraint: enforce residual = 0
    interior = PointwiseInteriorConstraint(
        nodes=nodes,
        geometry=geo,
        outvar={
            FIXME # Fill in: Add PDE constraints here
        },
        batch_size=cfg.batch_size.interior,
        parameterization=time_range,
    )
    domain.add_constraint(interior, "interior")

    # Validation data (decoupled exact solution)
    deltaT = 0.02
    deltaX = 0.06
    deltaY = 0.06
    x_vals = np.arange(0.0, L, deltaX)
    y_vals = np.arange(0.0, L, deltaY)
    t_vals = np.arange(0.0, 2 * L, deltaT)
    X, Y, TT = np.meshgrid(x_vals, y_vals, t_vals)
    X = np.expand_dims(X.flatten(), axis=-1)
    Y = np.expand_dims(Y.flatten(), axis=-1)
    TT = np.expand_dims(TT.flatten(), axis=-1)

    # Fill in: exact solutions
    Ta_true = FIXME # Fill in: Exact Ta
    To_true = FIXME # Fill in: Exact To

    invar_numpy = {"x": X, "y": Y, "t": TT}
    outvar_numpy = {"Ta": Ta_true, "To": To_true}

    validator = PointwiseValidator(
        nodes=nodes,
        invar=invar_numpy,
        true_outvar=outvar_numpy,
        batch_size=128,
        plotter=ValidatorPlotter(),
    )
    domain.add_validator(validator)

    slv = Solver(cfg, domain)
    slv.solve()
```

1. Fill in all missing parts
2. Place the configuration file in the conf directory
3. Run the script:

In [ ]:
!python climate_l2.py

--- 

Don't forget to check out additional [Open Hackathons Resources](https://www.openhackathons.org/s/technical-resources) and join our [OpenACC and Hackathons Slack Channel](https://www.openacc.org/community#slack) to share your experience and get more help from the community.

---

# Licensing

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials may include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.
